# G1 Academy Bonus - Task 9: low-level joint control + q/dq/kp/kd/tau visualizer

## Introduction
This task rebuilds the `rt/lowcmd` publisher lifecycle - message construction, CRC, freshness checking, and a single-joint `move_ll_joint` helper - then provides a small Jupyter UI (built with `ipywidgets` + `matplotlib`) so you can *feel* what `q`, `dq`, `kp`, `kd`, and `tau` each do: stream a commanded target continuously and plot commanded-vs-measured traces on demand.

`kp`/`kd` act as a virtual spring/damper pulling the joint toward the commanded `q`/`dq`; `tau` is a feed-forward torque added on top. High `kp` tracks position tighter but feels stiffer and can overshoot; `tau` alone with `kp=kd=0` is open-loop torque control with no position feedback at all.

In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1, ensure_channel_factory

ensure_channel_factory(0, "eth0")
g1 = G1("eth0")


## Task 1 - Native `rt/lowcmd` publisher: message construction, CRC, and defaults
One `LowCmd_` message is reused across writes; every one of the 29 body joints gets a full motor command (`mode`, `q`, `dq`, `tau`, `kp`, `kd`), and the message CRC is recomputed on every write before publishing - an unset or stale CRC is silently rejected by the firmware.

In [ ]:
# Task 2 imports the shared DDS initialization guard. Task 9 uses the
# shared wrapper rather than creating another ChannelFactory or publisher.
move_ll_joint = g1.move_ll_joint
get_lowstate = g1.get_lowstate

# Developer mode is an explicit operator action. These calls are deliberately
# not made by move_ll_joint or by the UI.
# g1.set_service("ai_sport", False)  # manually enter LowCmd developer mode
# g1.set_service("ai_sport", True)   # return AI Sport ownership


## Task 2 - `move_ll_joint(joint_id, q, dq, kp, kd, tau)`
Commands a single joint while holding every other joint at its currently observed position - the same pattern `sdk_wrapper.G1.move_ll_joint` uses for its `arm_sdk=False` path.

In [ ]:
# Normal control: rt/arm_sdk, joints 12-28, no service changes.
# move_ll_joint(22, 1.0, kp=20.0, kd=1.0)

# All-joint LowCmd control: only after the operator has manually disabled
# ai_sport, and without any service toggle from this helper.
# move_ll_joint(0, 0.1, kp=40.0, kd=1.0, dev_mode=True)


In [ ]:
# move_ll_joint(22, 1.0, kp=20.0, kd=1.0)  # rt/arm_sdk; AI Sport remains owner


## Task 3 - Jupyter UI: visualize q/dq/kp/kd/tau live
Sliders pick a joint and target `q`/`dq`/`kp`/`kd`/`tau`. "Stream command" starts a background thread that keeps calling `move_ll_joint` at a fixed rate with the current slider values (so you can tweak `kp`/`kd` live). "Capture & plot 3s" samples the *measured* `q`/`dq`/`tau_est` for the selected joint for three seconds and plots each against its commanded target - the gap between the dashed commanded line and the measured trace is exactly the tracking error `kp`/`kd` are fighting to close.

In [ ]:
import ipywidgets as widgets

joint_slider = widgets.IntSlider(min=12, max=28, value=22, description="joint_id")
q_slider = widgets.FloatSlider(min=-3.0, max=3.0, step=0.01, value=0.0, description="q target")
dq_slider = widgets.FloatSlider(min=-2.0, max=2.0, step=0.01, value=0.0, description="dq target")
kp_slider = widgets.FloatSlider(min=0.0, max=150.0, step=1.0, value=40.0, description="kp")
kd_slider = widgets.FloatSlider(min=0.0, max=5.0, step=0.05, value=1.0, description="kd")
tau_slider = widgets.FloatSlider(min=-5.0, max=5.0, step=0.05, value=0.0, description="tau (ff)")
dev_mode_toggle = widgets.Checkbox(value=False, description="LowCmd developer mode")
command_button = widgets.Button(description="Send command")
out = widgets.Output()

def _send_command(_button):
    with out:
        out.clear_output(wait=True)
        try:
            result = move_ll_joint(
                joint_slider.value, q_slider.value, dq_slider.value, kp_slider.value,
                kd_slider.value, tau_slider.value, dev_mode=dev_mode_toggle.value,
            )
            print(result)
        except Exception as exc:
            print(f"Command rejected: {exc}")

command_button.on_click(_send_command)
display(widgets.VBox([
    joint_slider, q_slider, dq_slider, kp_slider, kd_slider, tau_slider,
    dev_mode_toggle, command_button, out,
]))


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.